# GridIQ — 3-year vs 5-year window comparison (honest 24h-ahead)

Runs **XGBoost** and **Prophet** (Emma's config: flat growth, multiplicative seasonality, changepoint 0.01, + honest ≥24h demand lags) on **both** windows in one pass:

- **3-year**: train 2023–2024, test 2025
- **5-year**: train 2021–2024, test 2025

Same features, same honest day-ahead horizon — the only thing that changes is how many years of training data each model sees. This isolates the question: *did more years fix Prophet, or did the honest demand lags?* (Both models also benchmarked against ERCOT's day-ahead forecast.)

### Setup

In [ ]:
!pip install -q xgboost prophet pyarrow
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.ERROR); logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
import numpy as np, pandas as pd
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/GridIQ_Final_Project_MS_ADS_Spring_2026')
STAGING_PARQUET = DRIVE_ROOT / 'data' / 'staging' / 'staging_ercot_hourly.parquet'
TEMP_PARQUET    = DRIVE_ROOT / 'data' / 'staging' / 'ercot_temperature.parquet'
TEST_YEAR = 2025

### Load + weather (degree days)

In [ ]:
df = pd.read_parquet(STAGING_PARQUET)
df['utc_timestamp'] = pd.to_datetime(df['utc_timestamp'], utc=True)
df = df.sort_values('utc_timestamp').reset_index(drop=True)
w = pd.read_parquet(TEMP_PARQUET)
w['utc_timestamp'] = pd.to_datetime(w['utc_timestamp'], utc=True)
df = df.merge(w[['utc_timestamp','temp_c']], on='utc_timestamp', how='left')
df['temp_c'] = df['temp_c'].interpolate()
df['cdd'] = (df['temp_c'] - 18.0).clip(lower=0)
df['hdd'] = (18.0 - df['temp_c']).clip(lower=0)
print('loaded', len(df), 'rows', df.utc_timestamp.min(), '->', df.utc_timestamp.max())

### Models + the per-window runner

In [ ]:
import xgboost as xgb
from prophet import Prophet

REG = ['cdd','hdd','temp_c','demand_lag24','demand_lag168','demand_roll_w']
XFEAT = ['hour','dow','month','is_weekend','hour_sin','hour_cos','dow_sin','dow_cos',
         'lag_24','lag_168','temp_c','cdd','hdd']

def metrics(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    m = ~np.isnan(a) & ~np.isnan(p); a, p = a[m], p[m]
    return dict(MAE=np.mean(np.abs(a-p)), RMSE=np.sqrt(np.mean((a-p)**2)),
                MAPE_pct=np.mean(np.abs((a-p)/a))*100, rows=len(a))

def add_xfeat(d):
    d = d.copy()
    loc = d['utc_timestamp'].dt.tz_convert('US/Central')
    d['hour']=loc.dt.hour; d['dow']=loc.dt.dayofweek; d['month']=loc.dt.month
    d['is_weekend']=(d['dow']>=5).astype(int)
    d['hour_sin']=np.sin(2*np.pi*d['hour']/24); d['hour_cos']=np.cos(2*np.pi*d['hour']/24)
    d['dow_sin']=np.sin(2*np.pi*d['dow']/7);   d['dow_cos']=np.cos(2*np.pi*d['dow']/7)
    d['lag_24']=d['demand_mw'].shift(24); d['lag_168']=d['demand_mw'].shift(168)
    return d

def run_window(start, train_years):
    # window stays within its own years (lags computed inside the window)
    yr = df['utc_timestamp'].dt.year
    wdf = df[(yr >= start) & (yr <= TEST_YEAR)].copy().reset_index(drop=True)
    wdf['demand_lag24']  = wdf['demand_mw'].shift(24)
    wdf['demand_lag168'] = wdf['demand_mw'].shift(168)
    wdf['demand_roll_w'] = wdf['demand_mw'].shift(24).rolling(168, min_periods=24).mean()
    wy = wdf['utc_timestamp'].dt.year

    # XGBoost
    d = add_xfeat(wdf).dropna(subset=XFEAT + ['demand_mw']); dy = d['utc_timestamp'].dt.year
    tr = d[(dy >= train_years[0]) & (dy <= train_years[1])]; te = d[dy == TEST_YEAR]
    mx = xgb.XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
    mx.fit(tr[XFEAT], tr['demand_mw']); px = mx.predict(te[XFEAT])
    mxr = metrics(te['demand_mw'].values, px)

    # Prophet (Emma's config, honest lags)
    ptr = wdf[(wy >= train_years[0]) & (wy <= train_years[1])]; pte = wdf[wy == TEST_YEAR]
    def tp(fr):
        o = pd.DataFrame({'ds': fr['utc_timestamp'].dt.tz_localize(None)})
        for r in REG: o[r] = fr[r].values
        o['y'] = fr['demand_mw'].values
        return o.dropna()
    a, b = tp(ptr), tp(pte)
    mp = Prophet(growth='flat', daily_seasonality=True, weekly_seasonality=True,
                 yearly_seasonality=True, seasonality_mode='multiplicative',
                 changepoint_prior_scale=0.01)
    for r in REG: mp.add_regressor(r)
    mp.fit(a[['ds','y'] + REG]); fc = mp.predict(b[['ds'] + REG])
    mpr = metrics(b['y'].values, fc['yhat'].values)
    return mxr, mpr

print('runner ready')

### Run both windows + ERCOT benchmark

In [ ]:
WINDOWS = {'3-year (train 2023-24)': (2023, (2023, 2024)),
           '5-year (train 2021-24)': (2021, (2021, 2024))}

rows = []
for label, (start, ty) in WINDOWS.items():
    mxr, mpr = run_window(start, ty)
    rows.append(('XGBoost', label, mxr)); rows.append(('Prophet', label, mpr))
    print(f'{label}:  XGBoost {mxr["MAPE_pct"]:.2f}%   Prophet {mpr["MAPE_pct"]:.2f}%')

# ERCOT day-ahead benchmark (window-independent: actual vs ERCOT forecast on 2025)
te25 = df[df['utc_timestamp'].dt.year == TEST_YEAR]
erc = metrics(te25['demand_mw'].values, te25['demand_forecast_mw'].values)
rows.append(('ERCOT forecast', '— (day-ahead benchmark)', erc))

tbl = pd.DataFrame([{'model': m, 'window': w, **vals} for m, w, vals in rows])
tbl = tbl[['model','window','MAE','RMSE','MAPE_pct','rows']].round(2)
print(); print(tbl.to_string(index=False))

### Reading the table
If Prophet lands near ~4–5% on **both** windows, the takeaway is that the honest demand lags — not the number of years — are what carry it: the old 15% on 3 years came from running Prophet *without* demand lags. If the 3-year Prophet is clearly worse than the 5-year, then the extra history is doing real work. Either way you can now choose the window on its merits.